In [ ]:
!pip install -q transformers datasets evaluate rouge_score

In [ ]:
import torch
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import (
    BartTokenizer,
    BartForConditionalGeneration,
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq
)

In [ ]:
dataset = load_dataset("cnn_dailymail", "3.0.0")

train_data = dataset["train"].select(range(1300))  
val_data = dataset["validation"].select(range(100))

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
from transformers import BartTokenizer, BartForConditionalGeneration

model_name = "facebook/bart-large-cnn"

tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name).to(device)

In [ ]:
model.gradient_checkpointing_enable()

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

torch.cuda.empty_cache()

In [ ]:
def preprocess(example):
    model_inputs = tokenizer(
        example["article"],
        max_length=128,
        truncation=True
    )

    labels = tokenizer(
        example["highlights"],
        max_length=32,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


train_data = train_data.map(preprocess)
val_data = val_data.map(preprocess)

In [ ]:

rouge = evaluate.load("rouge")


In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # Sometimes predictions come as tuple
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    # If still logits → convert
    if predictions.ndim == 3:
        predictions = np.argmax(predictions, axis=-1)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )

    return result

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    fp16=True
)

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

In [ ]:
results = trainer.evaluate()
print(results)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

In [ ]:
def summarize(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)

    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=60,
        min_length=25,              # 🔥 smaller
        num_beams=5,
        length_penalty=2,         # 🔥 less aggressive
        no_repeat_ngram_size=3,
        repetition_penalty=1.8,
        early_stopping=True
    )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# Test
text = """
India secured a historic victory against Australia in the final match.
The team displayed exceptional batting and bowling performance.
Several players contributed significantly, leading to a dominant win.
"""

print(summarize(text))

In [ ]:
text = """
India achieved a remarkable victory against Australia in the final match of the series, showcasing exceptional performance across all departments. 
The batting lineup delivered a strong start, with key players forming crucial partnerships that built a solid foundation. 
The bowlers maintained consistent pressure, taking important wickets at critical moments and restricting the opposition's scoring opportunities. 
Fielding efforts were equally impressive, with sharp catches and quick ground coverage preventing easy runs. 
Several players stood out for their individual contributions, demonstrating both skill and teamwork throughout the match. 
This victory marks a significant milestone for the team and boosts their confidence ahead of upcoming international tournaments. 
The win also highlights the effectiveness of the team's strategy and preparation, reflecting the hard work put in by both players and coaching staff.
"""

print(summarize(text))

In [ ]:
text = """
Artificial Intelligence has rapidly transformed various industries, including healthcare, finance, and transportation. 
Machine learning models are now capable of analyzing vast amounts of data to make accurate predictions and assist in decision-making processes. 
In healthcare, AI systems help doctors diagnose diseases earlier and recommend personalized treatments. 
In finance, algorithms detect fraudulent transactions and optimize investment strategies. 
Despite its advantages, AI also raises concerns related to job displacement, data privacy, and ethical decision-making. 
Researchers and policymakers are actively working to ensure that AI development remains responsible and beneficial for society.
"""

print(summarize(text))

In [ ]:
text = """
A new startup has emerged in the electric vehicle market, aiming to revolutionize urban transportation with affordable and sustainable solutions. 
The company has developed innovative battery technology that allows faster charging and longer driving ranges. 
Investors have shown strong interest, leading to significant funding in early rounds. 
However, the startup faces challenges such as intense competition, regulatory hurdles, and the need for large-scale infrastructure. 
Experts believe that if executed well, the company could play a key role in accelerating the adoption of electric vehicles globally.
"""

print(summarize(text))